# M6A3 - Sistemas de Monitoramento de Experimentos

Na prática de hoje vamos monitorar um experimento utilizando o [TensorBoard](https://www.tensorflow.org/tensorboard?hl=pt-br).

Esse notebook está estruturado da seguinte forma.

- Introdução
- Rodar experimento simples
- Acompanhar logs
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [1]:
!pip install torch torchvision tensorboard

   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.5 MB ? eta -:--:--
   ---------------------------- ----------- 3.9/5.5 MB 14.8 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 16.3 MB/s  0:00:00
   ---------------------------------------- 0.0/5.1 MB ? eta -:--:--
   ---------------------------------------- 5.1/5.1 MB 27.9 MB/s  0:00:00

   ---------------------------------------- 0/7 [werkzeug]
   ---------------------------------------- 0/7 [werkzeug]
   ---------------------------------------- 0/7 [werkzeug]
   ---------------------------------------- 0/7 [werkzeug]
   ---------------------------------------- 0/7 [werkzeug]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------- ---------------------------- 2/7 [protobuf]
   ----------- ---------------------

Importar as bibliotecas

In [2]:
import torch
import torchvision
from torch.utils.tensorboard import SummaryWriter


## Rodar Experimentos Simples

Para isso iremos reproduzir o experimento da aula M4A2 sobre GANs.

In [3]:
####################
## Carregar Dados ##
####################

batch_size = 100

# MNIST Dataset
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.5), std=(0.5))])

train_dataset = torchvision.datasets.MNIST(root='./mnist_data/', train=True, transform=transform, download=True)
# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)


########################
## Criando os modelos ##
########################


# Gerador.
class Generator(torch.nn.Module):
    def __init__(self, g_input_dim, g_output_dim):
        super(Generator, self).__init__()       
        self.fc1 = torch.nn.Linear(g_input_dim, 256)
        self.fc2 = torch.nn.Linear(self.fc1.out_features, self.fc1.out_features*2)
        self.fc3 = torch.nn.Linear(self.fc2.out_features, self.fc2.out_features*2)
        self.fc4 = torch.nn.Linear(self.fc3.out_features, g_output_dim)
    
    # método forward. 
    def forward(self, x): 
        x = torch.nn.functional.leaky_relu(self.fc1(x), 0.2)
        x = torch.nn.functional.leaky_relu(self.fc2(x), 0.2)
        x = torch.nn.functional.leaky_relu(self.fc3(x), 0.2)
        return torch.tanh(self.fc4(x))

# Discrimador.
class Discriminator(torch.nn.Module):
    def __init__(self, d_input_dim):
        super(Discriminator, self).__init__()
        self.fc1 = torch.nn.Linear(d_input_dim, 1024)
        self.fc2 = torch.nn.Linear(self.fc1.out_features, self.fc1.out_features//2)
        self.fc3 = torch.nn.Linear(self.fc2.out_features, self.fc2.out_features//2)
        self.fc4 = torch.nn.Linear(self.fc3.out_features, 1)
    
    # método forward. 
    def forward(self, x):
        x = torch.nn.functional.leaky_relu(self.fc1(x), 0.2)
        x = torch.nn.functional.dropout(x, 0.3)
        x = torch.nn.functional.leaky_relu(self.fc2(x), 0.2)
        x = torch.nn.functional.dropout(x, 0.3)
        x = torch.nn.functional.leaky_relu(self.fc3(x), 0.2)
        x = torch.nn.functional.dropout(x, 0.3)
        return torch.sigmoid(self.fc4(x))
    

#################
## Treinamento ##
#################

# Instanciar o logger do Tensorboard.
# Pode alterar o path.
writer =  SummaryWriter("logs/gan_2")

# Instanciar as redes.
z_dim = 100
mnist_dim = train_dataset.train_data.size(1) * train_dataset.train_data.size(2)

device = "cuda" if torch.cuda.is_available() else "cpu"
G = Generator(g_input_dim = z_dim, g_output_dim = mnist_dim).to(device)
D = Discriminator(mnist_dim).to(device)

# Função de perda.
criterion = torch.nn.BCELoss() 

# Otimizador.
lr = 0.0002 
G_optimizer = torch.optim.Adam(G.parameters(), lr = lr)
D_optimizer = torch.optim.Adam(D.parameters(), lr = lr)

def D_train(x):
    #=======================Treino do discriminador=======================#
    D.zero_grad()

    # Treina discriminador em dados reais.
    x_real, y_real = x.view(-1, mnist_dim), torch.ones(batch_size, 1)
    x_real, y_real = torch.autograd.Variable(x_real.to(device)), torch.autograd.Variable(y_real.to(device))

    D_output = D(x_real)
    D_real_loss = criterion(D_output, y_real)
    D_real_score = D_output

    # Treina discriminador em dados falsos.
    z = torch.autograd.Variable(torch.randn(batch_size, z_dim).to(device))
    x_fake, y_fake = G(z), torch.autograd.Variable(torch.zeros(batch_size, 1).to(device))

    D_output = D(x_fake)
    D_fake_loss = criterion(D_output, y_fake)
    D_fake_score = D_output

    # Backpropagation e otimização dos parâmetros do discriminador.
    D_loss = D_real_loss + D_fake_loss
    D_loss.backward()
    D_optimizer.step()
        
    return  D_loss.data.item()

def G_train(x):
    #=======================Treino do gerador=======================#
    G.zero_grad()

    z = torch.autograd.Variable(torch.randn(batch_size, z_dim).to(device))
    y = torch.autograd.Variable(torch.ones(batch_size, 1).to(device))

    G_output = G(z)
    D_output = D(G_output)
    G_loss = criterion(D_output, y)

    # Backpropagation e otimização dos parâmetros do gerador.
    G_loss.backward()
    G_optimizer.step()
        
    return G_loss.data.item()

# Laço de treino.
n_epoch = 200
for epoch in range(1, n_epoch+1):           
    D_losses, G_losses = [], []
    for batch_idx, (x, _) in enumerate(train_loader):
        D_losses.append(D_train(x))
        G_losses.append(G_train(x))
        writer.add_scalar("Loss/Treino_Discriminator_step", D_losses[-1], batch_idx)
        writer.add_scalar("Loss/Treino_Generator_step", G_losses[-1], batch_idx)

    print('[%d/%d]: loss_d: %.3f, loss_g: %.3f' % (
            (epoch), n_epoch, torch.mean(torch.FloatTensor(D_losses)), torch.mean(torch.FloatTensor(G_losses))))
    writer.add_scalar("Loss/Treino_Discriminator_epoch", torch.mean(torch.FloatTensor(D_losses)), epoch)
    writer.add_scalar("Loss/Treino_Generator_epoch", torch.mean(torch.FloatTensor(G_losses)), epoch)
    
    # Gerando imagens para os logs.
    with torch.no_grad():
        test_z = torch.autograd.Variable(torch.randn(batch_size, z_dim).to(device))
        generated = G(test_z)

    generated = generated.view(generated.size(0), 1, 28, 28)
    grid = torchvision.utils.make_grid(generated.cpu(), 20)
    writer.add_image("Imagens Geradas", grid, epoch)

writer.flush()
writer.close()

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.55MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 147kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.42MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.6MB/s]
c:\Users\kaiog\Documents\nexvisual-computer-vision\modulos\.venv\Lib\site-packages\torchvision\datasets\mnist.py:76: UserWarning: train_data has been renamed data
  warnings.warn("train_data has been renamed data")


[1/200]: loss_d: 0.580, loss_g: 4.813
[2/200]: loss_d: 0.402, loss_g: 8.237
[3/200]: loss_d: 0.621, loss_g: 4.052
[4/200]: loss_d: 0.423, loss_g: 3.600
[5/200]: loss_d: 0.288, loss_g: 4.149
[6/200]: loss_d: 0.382, loss_g: 3.748
[7/200]: loss_d: 0.403, loss_g: 3.505
[8/200]: loss_d: 0.387, loss_g: 3.499
[9/200]: loss_d: 0.446, loss_g: 3.258
[10/200]: loss_d: 0.510, loss_g: 2.815
[11/200]: loss_d: 0.557, loss_g: 2.692
[12/200]: loss_d: 0.628, loss_g: 2.510
[13/200]: loss_d: 0.624, loss_g: 2.353
[14/200]: loss_d: 0.683, loss_g: 2.343
[15/200]: loss_d: 0.699, loss_g: 2.280
[16/200]: loss_d: 0.760, loss_g: 2.012
[17/200]: loss_d: 0.795, loss_g: 1.891
[18/200]: loss_d: 0.829, loss_g: 1.811
[19/200]: loss_d: 0.855, loss_g: 1.758
[20/200]: loss_d: 0.878, loss_g: 1.692
[21/200]: loss_d: 0.891, loss_g: 1.635
[22/200]: loss_d: 0.894, loss_g: 1.642
[23/200]: loss_d: 0.921, loss_g: 1.586
[24/200]: loss_d: 0.907, loss_g: 1.591
[25/200]: loss_d: 0.990, loss_g: 1.419
[26/200]: loss_d: 0.982, loss_g: 1

## Acompanhar Logs

Para isso rodamos o seguinte comando.

In [4]:
!tensorboard --logdir logs

^C


## Próximos Passos e Referências

E essa é a última prática da nossa trilha de Visão Computacional, espero que tenho aproveitado.

Uma lista não exaustiva de referências segue:

- https://www.tensorflow.org/tensorboard?hl=pt-br
- https://docs.pytorch.org/docs/stable/tensorboard.html
- https://github.com/lyeoni/pytorch-mnist-GAN
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Explore outras funções de logs possíveis no tensorboard e veja os logs.
- [ ] Existem outras soluções para controle de experimentos?

Histogramas dos pesos

In [5]:
for name, param in G.named_parameters():
    writer.add_histogram(f"Generator/{name}", param, epoch)

for name, param in D.named_parameters():
    writer.add_histogram(f"Discriminator/{name}", param, epoch)

Histogramas dos gradientes

In [6]:
for name, param in G.named_parameters():
    if param.grad is not None:
        writer.add_histogram(f"Gradients/G/{name}", param.grad, epoch)

- [ ] Existem outras soluções para controle de experimentos?

AimMLflow

In [19]:
%pip install mlflow

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.6 MB 14.5 MB/s eta 0:00:01
   -------------------------- ------------- 8.4/12.6 MB 25.3 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 24.0 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ------------------------------------ --- 3.1/3.5 MB 101.0 MB/s eta 0:00:01
   ------------------------------------ --- 3.1/3.5 MB 101.0 MB/s eta 0:00:01
   ------------------------------------ --- 3.1/3.5 MB 101.0 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 3.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 56.0 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 21.5 MB/s  0:00:00
   ---------------------------------

  You can safely remove it manually.


In [ ]:
import mlflow

mlflow.start_run()

mlflow.log_param("lr", lr)
mlflow.log_metric("generator_loss", g_loss)